## Оценка качества retriever на нескольких `top_k`

Этот ноутбук повторяет логику `questions_evaluation.ipynb`, но считает метрики для нескольких значений `top_k` в ретривере: **3, 5, 10, 15, 20**.

Что делает:
- загружает вопросы из `data/eval_questions_final.xlsx` (Sheet1 + Sheet2)
- чистит/фильтрует (`reason_skip`, `answerable`)
- для строк с `chunk_id` считает rank-метрики (MRR/Hit/Recall/NDCG/AUC) для каждого `top_k`
- (опционально) для строк без `chunk_id` делает LLM-оценку полезности retrieval
- строит сравнение метрик по `top_k` + сохраняет ECDF-графики по группам

Перед запуском:
- активируй `venv`
- проверь, что есть индекс/векторстор, который использует `app.rag.retriever.RetrieverFS`
- (опционально) выставь `OPENAI_API_KEY` для LLM-блока


In [ ]:
import os
import ast
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv

    load_dotenv()
    load_dotenv(".env.local", override=True)
except Exception:
    # dotenv не обязателен
    pass

DATA_PATH = Path("data") / "eval_questions_final.xlsx"
assert DATA_PATH.exists(), f"Не найден файл: {DATA_PATH.resolve()}"


def parse_list(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return parsed
        return [parsed]
    except Exception:
        return [str(val)]


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Сам_запрос" in df.columns:
        df = df[df["Сам_запрос"].astype(str).str.strip() != ""]

    for col in ["answerable", "question_type", "difficulty", "supporting_quote"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list)

    if "Ошибка_в_логике" in df.columns:
        df["Ошибка_в_логике"] = df["Ошибка_в_логике"].fillna("")

    return df.reset_index(drop=True)


# Загружаем оба листа (Sheet1/Sheet2)
sheets_raw = pd.read_excel(DATA_PATH, sheet_name=None)
sheets_raw = {
    name: df.assign(source_sheet=name, is_generated=(str(name).lower() == "sheet1"))
    for name, df in sheets_raw.items()
}

sheets_clean = {name: clean_df(df) for name, df in sheets_raw.items()}

for name, df in sheets_clean.items():
    flagged = (
        (df["Ошибка_в_логике"].astype(str).str.strip() != "").sum()
        if "Ошибка_в_логике" in df.columns
        else 0
    )
    print(f"Sheet: {name}, rows={len(df)}, flagged(Ошибка_в_логике): {flagged}")

combined_df = pd.concat(sheets_clean.values(), ignore_index=True)
combined_df["source"] = combined_df.apply(
    lambda r: "generated" if str(r.get("source_sheet", "")).lower() == "sheet1" else "selfmade",
    axis=1,
)

# убираем пропущенные (если колонка есть)
if "reason_skip" in combined_df.columns:
    before = len(combined_df)
    combined_df = combined_df[combined_df["reason_skip"].isna()].copy()
    print(f"Filtered reason_skip: {before} -> {len(combined_df)}")


def all_true_or_empty(val):
    if val == [] or (isinstance(val, float) and pd.isna(val)):
        return True
    return isinstance(val, list) and all(x is True for x in val)


filtered_df = combined_df[combined_df["answerable"].apply(all_true_or_empty)].copy()
print(f"rows: {len(filtered_df)} (from combined {len(combined_df)})")

# нормализуем значения
if "Профессиональный" in filtered_df.columns:
    filtered_df.loc[
        (filtered_df["Профессиональный"] == "isu") | (filtered_df["Профессиональный"] == "Профессионалы"),
        "Профессиональный",
    ] = "профессионалы"

filtered_df["query_id"] = range(len(filtered_df))

# артефакт для дебага
(Path("data")).mkdir(parents=True, exist_ok=True)
filtered_df.to_csv(Path("data") / "filtered_df.csv", index=False, encoding="utf-8")
filtered_df.head(3)


In [ ]:
import math

from app.rag.retriever import RetrieverFS

# ВАЖНО: считаем метрики для нескольких top_k
TOP_K_LIST = [3, 5, 10, 15, 20]
retriever_fs = RetrieverFS(top_k=max(TOP_K_LIST))


def norm_chunk_id(val):
    if pd.isna(val):
        return None
    try:
        if float(val).is_integer():
            return str(int(float(val)))
    except Exception:
        pass
    return str(val)


def extract_chunk_id(doc):
    for key in ["chunk_id", "id", "chunk", "chunkId"]:
        if key in doc.metadata and doc.metadata[key] is not None:
            return norm_chunk_id(doc.metadata[key])
    return None


has_chunk_df = filtered_df[filtered_df["chunk_id"].notna()].copy() if "chunk_id" in filtered_df.columns else filtered_df.iloc[0:0]
no_chunk_df = filtered_df[filtered_df["chunk_id"].isna()].copy() if "chunk_id" in filtered_df.columns else filtered_df.copy()

print(f"Всего в filtered_df: {len(filtered_df)}")
print(f"c chunk_id: {len(has_chunk_df)}, без chunk_id: {len(no_chunk_df)}")


In [ ]:
import re
import matplotlib.pyplot as plt
import seaborn as sns


def text_match_soft(targets, doc_text: str, min_overlap: float = 0.2) -> bool:
    if not targets:
        return False
    doc_low = (doc_text or "").lower()

    if any(t in doc_low for t in targets):
        return True

    words_doc = {w for w in re.findall(r"\w+", doc_low) if len(w) > 3}
    for t in targets:
        words_t = {w for w in re.findall(r"\w+", t) if len(w) > 3}
        if not words_t:
            continue
        overlap = len(words_doc & words_t) / len(words_t)
        if overlap >= min_overlap:
            return True
    return False


def build_labels(row, docs, rel_id=None, use_chunk_id: bool = True, soft: bool = False):
    labels = []

    truth_texts = []
    if not pd.isna(row.get("Ожидаемый_ответ", None)):
        truth_texts.append(str(row["Ожидаемый_ответ"]))
    if "supporting_quote" in row and isinstance(row["supporting_quote"], list):
        truth_texts.extend([str(x) for x in row["supporting_quote"] if x])
    lowered_truth = [t.lower() for t in truth_texts if t]

    for d in docs:
        lab = 0
        if use_chunk_id and rel_id is not None:
            cid = extract_chunk_id(d)
            if cid == rel_id:
                lab = 1

        if lab == 0 and lowered_truth:
            dtext = (d.page_content or "").lower()
            if soft:
                if text_match_soft(lowered_truth, dtext):
                    lab = 1
            else:
                if any(t in dtext for t in lowered_truth):
                    lab = 1

        labels.append(lab)

    return labels


def metrics_from_labels(labels):
    if not labels:
        return dict(rank=-1, MRR=0.0, Hit_rate=0.0, Recall=0.0, NDCG=0.0, AUC=0.0)

    try:
        rank = labels.index(1)
    except ValueError:
        rank = -1

    if rank == -1:
        return dict(rank=-1, MRR=0.0, Hit_rate=0.0, Recall=0.0, NDCG=0.0, AUC=0.0)

    k = len(labels)
    mrr = 1.0 / (rank + 1)
    hit = 1.0
    recall = 1.0
    ndcg = 1.0 / math.log2(rank + 2)
    auc = (k - rank) / k

    return dict(rank=rank, MRR=mrr, Hit_rate=hit, Recall=recall, NDCG=ndcg, AUC=auc)


def eval_rank_metrics(df: pd.DataFrame, retriever_fs: RetrieverFS, top_k_list=TOP_K_LIST):
    rows = []
    meta_cols = [c for c in df.columns if c not in {"Сам_запрос", "chunk_id"}]

    for k in top_k_list:
        retriever_fs.top_k = k
        for _, row in df.iterrows():
            query = row["Сам_запрос"]
            rel_id = norm_chunk_id(row["chunk_id"])
            docs = retriever_fs.vectorstore.similarity_search(query=query, k=k)

            labels = build_labels(row, docs, rel_id=rel_id, use_chunk_id=True)
            metrics = metrics_from_labels(labels)

            base_row = {
                "query": query,
                "chunk_id": rel_id,
                "k": k,
                "rank": metrics["rank"],
                "MRR": metrics["MRR"],
                "Hit_rate": metrics["Hit_rate"],
                "Recall": metrics["Recall"],
                "NDCG": metrics["NDCG"],
                "AUC": metrics["AUC"],
                "retrieved_chunk_ids": [extract_chunk_id(d) for d in docs],
                "retrieved_texts": [d.page_content for d in docs],
                "labels": labels,
                "origin": "has_chunk",
            }
            for c in meta_cols:
                base_row[c] = row.get(c)
            rows.append(base_row)

    per_query = pd.DataFrame(rows)
    summary = per_query.groupby("k")[["MRR", "Hit_rate", "Recall", "NDCG", "AUC"]].mean().reset_index()
    return per_query, summary


per_query_rank, summary_rank = eval_rank_metrics(has_chunk_df, retriever_fs, TOP_K_LIST)
print("Перезапросов с gt chunk_id:", len(has_chunk_df))
display(summary_rank)

# Сравнение по top_k (overall)
metric_cols = ["MRR", "Hit_rate", "Recall", "NDCG", "AUC"]
summary_long = summary_rank.melt(id_vars=["k"], value_vars=metric_cols, var_name="metric", value_name="value")
plt.figure(figsize=(8, 4))
sns.lineplot(data=summary_long, x="k", y="value", hue="metric", marker="o")
plt.title("Overall rank-metrics vs top_k (chunk_id queries)")
plt.xlabel("top_k")
plt.ylabel("mean")
plt.tight_layout()
plt.show()


In [ ]:
# Запросы без chunk_id: (опционально) оценка через LLM полезности retrieval
# ВАЖНО: это дорого/долго. По умолчанию выключено.

import textwrap
from tqdm import tqdm

RUN_LLM = False  # поставь True, если хочешь реально дергать LLM
RUN_LLM_MULTIK = True  # True -> прогон для каждого k из TOP_K_LIST

LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL")
TOP_K_LLM = 10  # используется только если RUN_LLM_MULTIK=False

if RUN_LLM:
    assert OPENAI_API_KEY, "Set OPENAI_API_KEY (например в PowerShell: $env:OPENAI_API_KEY='...')"

try:
    import openai
except Exception:
    openai = None


def llm_score_retrieval(query: str, expected_answer: str, docs_text: str, model: str = LLM_MODEL) -> float:
    """0..1 насколько извлечённые фрагменты помогают ответить."""
    client = openai.OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL)
    system = "Ты проверяешь, насколько извлечённые фрагменты помогут ответить на вопрос. Выдай только число от 0 до 1."
    user = textwrap.dedent(
        f"""
        Вопрос: {query}
        Ожидаемый ответ: {expected_answer}
        Извлечённые фрагменты:
        {docs_text}
        Оцени, даст ли это правильно ответить. Верни только число 0..1.
        """
    ).strip()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0,
        max_tokens=10,
    )
    txt = resp.choices[0].message.content.strip()
    try:
        return float(txt)
    except Exception:
        return 0.0


def llm_label_docs(query: str, expected_answer: str, docs: list[str], model: str = LLM_MODEL) -> list[int]:
    """Просим LLM вернуть список 0/1 по каждому чанку."""
    if not docs:
        return []

    client = openai.OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL)
    numbered = "\n\n".join([f"[{i}] {t[:800]}" for i, t in enumerate(docs)])
    system = "Оцени каждый фрагмент: 1 если он помогает ответить на вопрос, 0 если нет. Верни JSON массив нулей и единиц без текста."
    user = textwrap.dedent(
        f"""
        Вопрос: {query}
        Ожидаемый ответ: {expected_answer}
        Фрагменты:
        {numbered}
        Верни JSON список из {len(docs)} чисел 0/1 в том же порядке, без пояснений.
        """
    ).strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0,
            max_tokens=120,
        )
        txt = resp.choices[0].message.content.strip()
        import json

        try:
            arr = json.loads(txt)
        except Exception:
            nums = re.findall(r"[01]", txt)
            arr = [int(x) for x in nums]

        arr = [int(x) for x in arr if str(x) in {"0", "1"}]
        if len(arr) < len(docs):
            arr += [0] * (len(docs) - len(arr))
        return arr[: len(docs)]
    except Exception:
        return [0] * len(docs)


llm_rows = []
meta_cols = [c for c in filtered_df.columns if c not in {"Сам_запрос", "chunk_id"}]

if RUN_LLM and openai is None:
    raise RuntimeError("openai package is not available but RUN_LLM=True")

if RUN_LLM:
    k_values = TOP_K_LIST if RUN_LLM_MULTIK else [TOP_K_LLM]

    for k in k_values:
        for _, row in tqdm(no_chunk_df.iterrows(), total=len(no_chunk_df), desc=f"LLM eval (k={k})"):
            query = row["Сам_запрос"]
            expected = row.get("Ожидаемый_ответ", "")

            docs = retriever_fs.vectorstore.similarity_search(query=query, k=k)

            llm_labels = llm_label_docs(query, expected, [d.page_content for d in docs], model=LLM_MODEL)
            metrics = metrics_from_labels(llm_labels)

            docs_text = "\n---\n".join([d.page_content[:800] for d in docs])
            score = llm_score_retrieval(query, expected, docs_text, model=LLM_MODEL)

            base_row = {
                "query": query,
                "k": k,
                "score": score,
                "labels": llm_labels,
                "rank": metrics["rank"],
                "MRR": metrics["MRR"],
                "Hit_rate": metrics["Hit_rate"],
                "Recall": metrics["Recall"],
                "NDCG": metrics["NDCG"],
                "AUC": metrics["AUC"],
                "retrieved": len(docs),
                "retrieved_texts": [d.page_content for d in docs],
                "origin": "llm_no_chunk",
            }
            for c in meta_cols:
                base_row[c] = row.get(c)
            llm_rows.append(base_row)

    llm_eval_df = pd.DataFrame(llm_rows)
    print("LLM оценило строк:", len(llm_eval_df))
    display(llm_eval_df.head())
else:
    llm_eval_df = pd.DataFrame()
    print("RUN_LLM=False -> LLM-оценка пропущена")


In [ ]:
# Объединяем результаты (has_chunk + llm_no_chunk)
frames = []
if "per_query_rank" in globals() and isinstance(per_query_rank, pd.DataFrame) and not per_query_rank.empty:
    frames.append(per_query_rank)
if "llm_eval_df" in globals() and isinstance(llm_eval_df, pd.DataFrame) and not llm_eval_df.empty:
    frames.append(llm_eval_df)

all_eval_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("Всего строк в объединённой оценке:", len(all_eval_df))
if not all_eval_df.empty:
    display(all_eval_df.head())

# сохраняем артефакт
(Path("data")).mkdir(parents=True, exist_ok=True)
all_eval_df.to_csv(Path("data") / "all_eval_df_multi_topk.csv", index=False, encoding="utf-8")


In [ ]:
# Сводка overall: метрики по k (+ origin если есть)
assert isinstance(all_eval_df, pd.DataFrame), "Нет all_eval_df"

metric_cols = ["MRR", "Hit_rate", "Recall", "NDCG", "AUC"]

if all_eval_df.empty:
    print("all_eval_df пустой")
else:
    overall_by_k = all_eval_df.groupby("k")[metric_cols].mean().reset_index().sort_values("k")
    display(overall_by_k)

    plt.figure(figsize=(8, 4))
    overall_long = overall_by_k.melt(id_vars=["k"], value_vars=metric_cols, var_name="metric", value_name="value")
    sns.lineplot(data=overall_long, x="k", y="value", hue="metric", marker="o")
    plt.title("Overall metrics vs top_k")
    plt.xlabel("top_k")
    plt.ylabel("mean")
    plt.tight_layout()
    plt.show()

    if "origin" in all_eval_df.columns:
        by_origin_k = all_eval_df.groupby(["origin", "k"])[metric_cols].mean().reset_index().sort_values(["origin", "k"])
        display(by_origin_k)

        for m in metric_cols:
            plt.figure(figsize=(7, 3.5))
            sns.lineplot(data=by_origin_k, x="k", y=m, hue="origin", marker="o")
            plt.title(f"{m} vs top_k (by origin)")
            plt.xlabel("top_k")
            plt.ylabel(m)
            plt.tight_layout()
            plt.show()


In [ ]:
# ECDF распределения метрик по группам + сохранение
from pathlib import Path

out_root = Path("data/figures/ecdf_multi_topk")
out_root.mkdir(parents=True, exist_ok=True)

if all_eval_df.empty:
    print("all_eval_df пустой -> ECDF пропущен")
else:
    group_cols = [
        "is_generated",
        "Вид_катания",
        "Профессиональный",
        "Интернациональный/РФ",
        "Кол-во_вопросов_в_запросе",
    ]

    # какие k реально есть в данных
    k_values = sorted([int(x) for x in all_eval_df["k"].dropna().unique()])

    for top_k in k_values:
        current_df = all_eval_df.loc[all_eval_df["k"] == top_k].copy()

        for gc in group_cols:
            if gc not in current_df.columns:
                continue

            df_gc = current_df.dropna(subset=metric_cols, how="all")
            if df_gc.empty:
                continue

            n_total = len(df_gc)
            counts = df_gc[gc].value_counts().to_dict()
            hue_order = [k for k, _ in sorted(counts.items(), key=lambda x: -x[1])]
            palette = dict(zip(hue_order, sns.color_palette("tab10", n_colors=len(hue_order))))

            group_dir = out_root / gc.replace("/", "_")
            group_dir.mkdir(parents=True, exist_ok=True)

            for m in metric_cols:
                if df_gc[m].dropna().empty:
                    continue

                plt.figure(figsize=(7, 4))
                sns.ecdfplot(
                    data=df_gc,
                    x=m,
                    hue=gc,
                    hue_order=hue_order,
                    palette=palette,
                    linewidth=2,
                    alpha=0.9,
                )
                groups_n = ", ".join([f"{k}: n={v}" for k, v in counts.items()])
                plt.title(f"ECDF: {gc} | {m} (n_total={n_total}) | {groups_n} | top_k={top_k}")
                plt.xlabel(m)
                plt.ylabel("ECDF")
                plt.tight_layout()

                fname = f"ecdf_{gc.replace('/', '_')}_{m}_top_k_{top_k}.png".replace(" ", "_")
                plt.savefig(group_dir / fname, dpi=200)
                plt.show()
